## Setup 

This section is for setting up the python environment. 

It loads a few python libraries we'll be using.

You can just run it and move on! 

In [16]:
# Pandas and NumPy settings for data analysis
import pandas as pd
import numpy as np

# Always show all dataframe columns
pd.set_option('display.max_columns', None)

# Some libraries for better displaying in Jupyter
from IPython.display import display, HTML

# TQDM for progress bars
from tqdm.notebook import tqdm
tqdm.pandas()

In [17]:
import os
import openai
import dotenv
dotenv.load_dotenv()

openai.organization = None
openai.api_key = os.getenv("OPENAI_API_KEY")
# openai.Model.list() # see all openai models

## Load Data

In this section I show the ouptput of some data we grabbed from MediaCloud.

In [ ]:
 # Load bill summaries for Maryland bills
stories = pd.read_csv('stories_df.csv')

# Display the stories
stories

# Keywords

This section extracts some keywords from the bills using a library called `yake` (Yet Another Keyword Extractor)

It also uses `pandarallel` to parallelize this process and get it done faster!

In [ ]:
from yake import KeywordExtractor
from pandarallel import pandarallel

# Define a function to extract ketwords from a text
kw_extractor = KeywordExtractor()
def get_keywords(text):
    keywords = kw_extractor.extract_keywords(text)
    return [x for x,y in keywords]

# Run the function in parallel to speed things up
pandarallel.initialize(progress_bar=True)
stories['keywords'] = stories['snippet'].parallel_apply(get_keywords)

# display the stories with the keywords attatched
stories

## Remove documents that are too big

OpenAI models have a context window of ~8000 tokens. 

https://platform.openai.com/docs/guides/embeddings/embedding-models

So we'll remove any documents that are longer than that for now.

We can use `tiktoken` to [count the tokens](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb) in each document.

In [ ]:
# Import modules
import tiktoken
from openai import OpenAI
client = OpenAI()

# Set embedding model parameters
embedding_model = "text-embedding-3-small" # this is the model we will use to make embeddings
embedding_encoding = "cl100k_base"  # this the encoding for text-embedding-ada-002
max_tokens = 8000  # the maximum for text-embedding-ada-002 is 8191

# Get the encoding for the specified model
encoding = tiktoken.get_encoding(embedding_encoding)

# Make a new column with the combined title and summary
stories["combined"] = (
    "Title: " + stories.title.str.strip() + "; Content: " + stories.snippet.str.strip()
)

# Make a new column with the number of tokens in the combined title and summary
stories["n_tokens"] = stories.combined.apply(lambda x: len(encoding.encode(x)))

# Sort by that column
stories = stories.sort_values(by='n_tokens', ascending=False)

# Display the stories
stories


In [ ]:
# Grab the rows where the text is too big for the context window of the mmodel (>8000 tokens)
too_long = stories.query("n_tokens > @max_tokens") 

# Print how many will be removed
print(f"Removing {len(too_long)} stories that are too long")

# Display the removed stories here in this cell so we can see what we're losing
display(too_long)  

# Remove the rows where the text is too big for the context window of the model
stories = stories.query("n_tokens <= @max_tokens")  

## Embeddings

Now we take the "combined" column, which contains the 

In [ ]:
from openai import OpenAI
client = OpenAI()

def get_embeddings(texts, model="text-embedding-3-small"):
    # Replace newlines in each text and ensure it's a list of texts
    texts = [text.replace("\n", " ") for text in texts]
    # OpenAI's embeddings.create can process multiple inputs as a list
    response = client.embeddings.create(input=texts, model=model)
    # Extract embeddings from the response
    embeddings = [item.embedding for item in response.data]
    return embeddings

# Function to process DataFrame in batches and return a list of embeddings
def process_in_batches(df, column_name, batch_size=10):
    # Break the DataFrame into batches of size `batch_size`
    batches = [df[column_name].iloc[i:i + batch_size] for i in range(0, len(df), batch_size)]
    # Process each batch and collect embeddings
    all_embeddings = []
    for batch in tqdm(batches, desc="Processing batches"):
        batch_embeddings = get_embeddings(batch.tolist())
        all_embeddings.extend(batch_embeddings)
    return all_embeddings

# Example usage
batch_size = 100  # Adjust based on your preference and rate limits
stories['embedding'] = process_in_batches(stories, 'combined', batch_size=batch_size)


In [23]:
# drop combined column since those were only for the purposes of making the embeddings
stories = stories.drop(columns=['combined', 'n_tokens'])

## Dimensionality Reduction (t-SNE)

The embedding is a vector in 1536 dimensions. Viewing data in that many dimensions would break your brain. 🤯

Our brains can only handle 2 or 3 dimensions at a time. We'll use t-SNE to reduce the number of dimensions, flattening the multi-dimensional space into 2 dimensions.

> Here are some things to keep in mind about t-SNE if you use it in the future. You may have to tweak some parameters to fit the needs of your data.
>
> Blog Post: [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/)



In [ ]:
# find the one where bill.embedding is nan
stories[stories.embedding.isna()]

In [25]:
# remove where embedding is na 
stories = stories.dropna(subset=['embedding'])

In [26]:
from sklearn.manifold import TSNE
import numpy as np


# Convert to a list of lists of floats
matrix = np.array(stories.embedding.to_list())

# Create a t-SNE model and transform the data
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='random', learning_rate=400)
vis_dims = tsne.fit_transform(matrix)

# add to dataframe and write to csv
stories = stories\
    .assign(
        x = vis_dims[:,0], 
        y = vis_dims[:,1])


In [ ]:
# Write the data to a CSV file
stories.to_csv('../stories-with-embeddings.csv', index=False)
stories.drop(columns=['snippet', 'archive_playback_url', 'original_capture_url', 'embedding']).to_csv('../stories-no-snippet.csv', index=False)

# Display the bills
stories.head()